In [ ]:
!pip -q install -U bitsandbytes --no-deps
!pip -q install -U accelerate peft datasets scikit-learn


In [3]:
import os, random, json
import numpy as np
import pandas as pd
import torch, torch.nn.functional as F

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from datasets import Dataset

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
    DataCollatorWithPadding, TrainingArguments, Trainer, set_seed
)
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model

# Kaggle Secrets → HF token
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HUGGINGFACE_TOKEN")
assert HF_TOKEN, "Add HUGGINGFACE_TOKEN in Kaggle Settings → Secrets."

# Env tuning
os.environ.setdefault("HF_HOME", "/kaggle/working/hf_home")
os.environ.setdefault("HF_HUB_CACHE", "/kaggle/working/hf_home/hub")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")

# Seed
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); set_seed(SEED)

import transformers
print("Transformers version:", transformers.__version__)


2025-09-16 13:02:11.031803: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758027731.057597     158 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758027731.065709     158 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Transformers version: 4.52.4


In [4]:
TEXT_COL, LABEL_COL = "Review", "Label"
PREP_DIR = "/kaggle/input/fakereview/Dataset/prepared_paper_20250915_161745"  # change if needed

train_df = pd.read_csv(f"{PREP_DIR}/train.csv")
val_df   = pd.read_csv(f"{PREP_DIR}/val.csv")
test_df  = pd.read_csv(f"{PREP_DIR}/test.csv")

for df,n in [(train_df,"train"), (val_df,"val"), (test_df,"test")]:
    assert TEXT_COL in df.columns and LABEL_COL in df.columns, f"{n} missing {TEXT_COL}/{LABEL_COL}"

print(f"✅ Prepared splits loaded | train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")


✅ Prepared splits loaded | train=10004, val=1251, test=1251


In [5]:
MODEL_NAME = "BanglaLLM/Bangla-s1k-llama-3.2-3B-Instruct"
MAX_LEN = 256

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("⏳ Loading model…")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN,
)
print("✅ Model loaded.")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True, token=HF_TOKEN)
    print("✅ Loaded fast tokenizer.")
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False, trust_remote_code=True, token=HF_TOKEN)
    print("ℹ️ Loaded slow tokenizer.")

tokenizer.padding_side = "right"
ADDED_PAD = False
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({"pad_token": "<pad>"})
        ADDED_PAD = True
model.config.pad_token_id = tokenizer.pad_token_id
if ADDED_PAD:
    model.resize_token_embeddings(len(tokenizer))

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model = get_peft_model(model, lora_cfg)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.print_trainable_parameters()


⏳ Loading model…


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at BanglaLLM/Bangla-s1k-llama-3.2-3B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded.
✅ Loaded fast tokenizer.
trainable params: 4,593,664 || all params: 3,217,349,632 || trainable%: 0.1428


In [6]:
def tok(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True, max_length=MAX_LEN,
        padding=False, return_attention_mask=True,
    )

train_ds = Dataset.from_pandas(train_df).map(tok, batched=True).rename_column(LABEL_COL, "labels")
val_ds   = Dataset.from_pandas(val_df).map(tok, batched=True).rename_column(LABEL_COL, "labels")
test_ds  = Dataset.from_pandas(test_df).map(tok, batched=True).rename_column(LABEL_COL, "labels")

for ds in [train_ds, val_ds, test_ds]:
    ds.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    pr, rc, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": float(acc), "precision": float(pr), "recall": float(rc), "f1": float(f1)}


Map:   0%|          | 0/10004 [00:00<?, ? examples/s]

Map:   0%|          | 0/1251 [00:00<?, ? examples/s]

Map:   0%|          | 0/1251 [00:00<?, ? examples/s]

In [8]:
# %% 6) TrainingArguments (robust saving + auto-resume friendly)
OUTPUT_DIR = "/kaggle/working/fake_review_ckpts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=False,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    remove_unused_columns=False,
    max_grad_norm=1.0,
)

from transformers import TrainerCallback

class EvalAtEpochEnd(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        control.should_evaluate = True
        return control

# %% 7) Trainer
# Optional: add readable label mapping
model.config.id2label = {0: "Fake", 1: "Non-Fake"}
model.config.label2id = {"Fake": 0, "Non-Fake": 1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,   # <-- tokenizer=tokenizer এর বদলে
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # ⚠️ label_names বাদ দেয়া হয়েছে (আপনার ভার্সনে নেই)
)

# callback add *after* trainer is created
trainer.add_callback(EvalAtEpochEnd())


No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [9]:
# %% 8) Auto-resume logic
# Try working dir first (current run). If not found, optionally try an external read-only dataset.
# To resume from a persisted dataset, set CKPT_INPUT_DIR to something like "/kaggle/input/your-ckpt-dataset"
CKPT_INPUT_DIR = os.environ.get("CKPT_INPUT_DIR", "").strip()  # optional

last_ckpt = None
# A) last ckpt in OUTPUT_DIR (if this is a rerun in same session)
if os.path.isdir(OUTPUT_DIR):
    try:
        last_ckpt = get_last_checkpoint(OUTPUT_DIR)
    except Exception:
        last_ckpt = None

# B) if none, try external read-only dataset path
if (last_ckpt is None) and CKPT_INPUT_DIR and os.path.isdir(CKPT_INPUT_DIR):
    try:
        last_ckpt = get_last_checkpoint(CKPT_INPUT_DIR)
    except Exception:
        last_ckpt = None

if last_ckpt:
    print(f"🔁 Resuming from: {last_ckpt}")
else:
    print("🆕 No checkpoint found, training from scratch.")


🆕 No checkpoint found, training from scratch.


In [10]:
# %% 9) Train & Save final
trainer.train(resume_from_checkpoint=bool(last_ckpt))

final_dir = "/kaggle/working/final"
os.makedirs(final_dir, exist_ok=True)
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"✅ Final model saved to: {os.path.abspath(final_dir)}")

✅ Final model saved to: /kaggle/working/final


In [12]:
# %% 10) Evaluate (Validation + Test) & save artifacts
# Validation (final)
val_eval = trainer.evaluate(eval_dataset=val_ds)
print("📊 Validation:", val_eval)

# Test
pred_out = trainer.predict(test_ds)
logits = pred_out.predictions
y_pred = np.argmax(logits, axis=1)
y_true = test_df[LABEL_COL].to_numpy(dtype=int)
probs  = F.softmax(torch.tensor(logits), dim=1).cpu().numpy()

pred_df = pd.DataFrame({
    "Review": test_df[TEXT_COL].tolist(),
    "True": y_true,
    "Pred": y_pred,
    "Prob_Fake(0)": probs[:,0],
    "Prob_NonFake(1)": probs[:,1],
})
pred_df.to_csv("/kaggle/working/test_predictions_with_probs.csv", index=False, encoding="utf-8")

metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision_weighted": float(precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)[0]),
    "recall_weighted": float(precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)[1]),
    "f1_weighted": float(precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)[2]),
}
with open("/kaggle/working/metrics_test.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

report_str = classification_report(
    y_true, y_pred, labels=[0,1],
    target_names=["Fake(0)", "Non-Fake(1)"], digits=4, zero_division=0
)
print("\n🔎 Test Classification Report:\n", report_str)
with open("/kaggle/working/classification_report_test.txt", "w", encoding="utf-8") as f:
    f.write(report_str)

cm = confusion_matrix(y_true, y_pred, labels=[0,1])
pd.DataFrame(cm, index=["True_Fake(0)","True_NonFake(1)"], columns=["Pred_Fake(0)","Pred_NonFake(1)"]).to_csv("/kaggle/working/confusion_matrix_test.csv", encoding="utf-8")

mis_df = pred_df[pred_df["True"] != pred_df["Pred"]]
mis_df.to_csv("/kaggle/working/misclassified_cases_test.csv", index=False, encoding="utf-8")

print("✅ Saved artifacts in /kaggle/working : final/, test_predictions_with_probs.csv, metrics_test.json, classification_report_test.txt, confusion_matrix_test.csv, misclassified_cases_test.csv")


📊 Validation: {'eval_loss': 0.07548972219228745, 'eval_accuracy': 0.9840127897681854, 'eval_precision': 0.9840176973551358, 'eval_recall': 0.9840127897681854, 'eval_f1': 0.9840127284750532, 'eval_runtime': 298.1551, 'eval_samples_per_second': 4.196, 'eval_steps_per_second': 0.527, 'epoch': 3.0}

🔎 Test Classification Report:
               precision    recall  f1-score   support

     Fake(0)     0.9674    0.9952    0.9811       626
 Non-Fake(1)     0.9951    0.9664    0.9805       625

    accuracy                         0.9808      1251
   macro avg     0.9812    0.9808    0.9808      1251
weighted avg     0.9812    0.9808    0.9808      1251

✅ Saved artifacts in /kaggle/working : final/, test_predictions_with_probs.csv, metrics_test.json, classification_report_test.txt, confusion_matrix_test.csv, misclassified_cases_test.csv
